# Training Research Notebook

**PR 43 · Step 2A.** Verify frozen inputs and replay the historical abstention search. All comparisons use the PR 39 random split. Target-derived features can use later contacts’ labels, and the MLP early-stopping validation shared encoder fitting. Forward-in-time performance requires separate temporal validation.

In [ ]:
# CELL 1: RESEARCH SETUP
import sys
from pathlib import Path

here = Path.cwd().resolve()
bank_root = next(p for p in (here, *here.parents)
                 if (p / "feature_research/training_inputs.py").is_file())
sys.path.insert(0, str(bank_root))

from feature_research.training_side import draw_training_side
from feature_research.training_inputs import frozen_inputs, record_inputs

## Training population

Draw the accepted stratified split using row identifiers. Only training features and labels are selected. The complement's row identifiers contribute to the router split fingerprint. The Stage I/O split fingerprint is attested from PR 39.

In [ ]:
# CELL 2: TRAINING POPULATION
TRAIN = draw_training_side(indices_only=True)

## Frozen OOF inputs and reference configuration

Stages 1 and 3 provide positive-subscription probabilities (sigmoid at Stage 1, fold-gated isotonic at Stage 3). Stage 3 raw scores are retained separately. Each router supplies its original route, confidence, transformed score, and producing fold's base rate. On Pass 1 confidence measures negative precision, on Pass 2 positive precision. GLASS uses the first matching rule's fit-sample precision; RF uses inner-OOF band precision. On deferral the score equals the fold base rate and confidence is zero. Deferral carries no router vote; Meta-XGBoost masks the probability in this state.

PR 39 GLASS voter cuts target 0.70 recall and use hybrid weights (blend 0.5). Meta-XGBoost uses raw meta OOF probabilities and fold-nested class cuts. The replay fixes the accepted configurations and calls the production abstention search: retain confidence ≥ cut, require coverage ≥ 0.50, maximize retained F2 across distinct observed confidences, and choose the smaller cut on ties. The older research helper scans a fixed grid. These historical settings are references for later registration.

In [ ]:
# CELL 3: VERIFY UPSTREAM INPUTS AND HISTORICAL SEARCH
OOF, evidence = frozen_inputs(TRAIN)
for arm, table in OOF.items():
    print(arm, table.shape, table.route.value_counts().to_dict())
for name, reference in evidence["historical_reference"].items():
    print(name, reference["configuration"], reference["replay"])

## Research record

Save verified identities, accepted configurations, and replay results. Per-row tables remain in memory for later research. Step 2B registers policy and Step 2C adds nested Stage 4 validation.

In [ ]:
# CELL 4: WRITE STEP 2A EVIDENCE
record_path = record_inputs(evidence)
print(record_path)

## Step 2B — Register the Stage 4 decision policy

After the setup cell, run Cell 5 directly to register against the accepted Step 2A record. Cells 2–4 replay the historical inputs and are unnecessary for this step. The versioned policy below defines the research costs, coverage obligations, comparisons, budgets, and evidence required for promotion. The receipt ties its digest to the verified frozen-input record.

In [ ]:
# CELL 5: REGISTER THE STAGE 4 POLICY
from pprint import pprint
from feature_research.stage4_policy import load_policy, register

policy = load_policy()
registration_path = register()
print(registration_path)
pprint(policy, sort_dicts=False)

## Step 2C — Nested Stage 4 references on fixed upstream inputs

Run after Cells 1–3; Cells 4–5 need not be repeated. The five outer folds are the accepted shared producer folds. Inside each outer training partition, three stratified inner folds (seed 42) produce the predictions that select every learned Stage 4 quantity: GLASS voter cuts and weights, the Meta-XGBoost proposal (the 12 registered RandomSampler proposals, historical F2-at-0.5 objective, raw space), class thresholds and deferral cuts. The selected procedure is refitted on the outer training partition, locked, and scores the untouched outer rows once. Each procedure runs under the historical rule (retained F2, coverage ≥ 0.50) and the registered whole-system policy. The accepted PR 39 Meta-XGBoost hyperparameters are a descriptive reference only.

Before any search, the run verifies the policy, its receipt and the Step 2A record, and reproduces the accepted PR 39 cuts, weights and meta OOF probabilities from the same inputs. Optimism is outer achieved loss minus inner selected loss, weighted by outer rows. The record lists the digest of every stored score; the scores themselves (`nested_predictions.npz`, compressed) stay local beside it, for reuse in Step 4B.

Upstream OOF inputs stay fixed, and the upstream fits behind a row's inputs can include other outer rows' labels. This is nested Stage 4 evaluation, not nested cascade evaluation (PR 44). Inner-to-outer differences also reflect sampling variation and fitting-set size. The evidence is random-split and training-only; temporal validation is PR 49.

In [ ]:
# CELL 6: NESTED STAGE 4 REFERENCES
from feature_research.stage4_nested import nested_references, record_nested, summarize

NESTED, nested_evidence, nested_local = nested_references(OOF, evidence)
display(summarize(nested_evidence).round(4))
nested_paths = record_nested(nested_evidence, nested_local)
print(*nested_paths, sep="\n")

## Commit 3 — Voter diagnostics and GLASS decision structure

Run the diagnostic command once, set `PR43_VOTER_RUN` to its run ID, and execute Cell 7 after setup. The first tables describe the frozen PR 39 GLASS operating point. Compare the two fold-refitted rules separately. A sole correct voter is counted among active votes. A correct lone dissent requires all three. Exact flag budgets preserve whole ties.

The negative-margin ceiling takes negative LR/EBM probabilities to zero with all votes and positive evidence held fixed. A forced positive remains positive even at that ceiling. Calibration lines use a joint intercept and slope. Each optional route variant requires a qualifying contrast on its own outer training rows. Row-level uncertainty is conditional on the frozen random split.

In [ ]:
# CELL 7: COMMIT 3 VOTER DIAGNOSTICS
import os
import pandas as pd
from feature_research.research_records import read_run_record
from feature_research.stage4_policy import LOGS

report = read_run_record(os.environ['PR43_VOTER_RUN'], 'voter_diagnostics', root=LOGS)
hist = report['historical_glass']
display(pd.DataFrame(hist['contingency']['joint']))
for name in ('pairs', 'unique'):
    display(pd.DataFrame(hist['contingency'][name]).query("label == 'all' and route == 'all'"))
display({k: v['brackets'] for k, v in hist['matched'].items()}, hist['router_tradeoffs'])
display(pd.DataFrame(report['deferred_quality']), pd.DataFrame(report['calibration']))
comp = pd.DataFrame(hist['composition'])
display(comp.groupby(['route','y','retained','correct']).n.sum().unstack(fill_value=0))
display(pd.DataFrame(hist['margin_bound']), pd.DataFrame(report['nested_glass_summary']))
comp.groupby(['route','retained']).n.sum().unstack(fill_value=0).plot.bar(stacked=True, ylabel='Training rows')
display(pd.DataFrame(report['route_coefficients'])[['upstream','outer_fold','status']])
display(report['handoff'])
print(report['rule_evidence'], report['joint_corners'], report['commit5_expansion'], sep='\n')